### Imports

In [2]:
import json
import re
import ast
from tqdm import tqdm
import tokenize
from io import StringIO
import ast
import inspect


### Load parsed data

In [4]:
# Load parsed data from previous step
INPUT_PATH = "../data/processed/parsed_data.json"

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    parsed_data = json.load(f)

print(f"Loaded {len(parsed_data)} files")

Loaded 291 files


### Text cleaning

In [6]:
def clean_text(text):
    """
    Basic text cleaning:
    - remove extra spaces
    - normalize newlines
    """
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

### Extract meaning from Python code

In [8]:

def extract_python_semantics_advanced(code):
    """
    Extracts rich information from Python code:
    - Function names + signatures (parameters)
    - Docstrings
    - Class names + methods
    - Imports (what libraries are used)
    - Constants and important variables
    - Return types (if annotated)
    - Decorators
    """
    try:
        tree = ast.parse(code)
    except Exception:
        return ""

    results = []
    
    # 1. Extract imports
    imports = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                imports.append(f"Import: {alias.name}")
        elif isinstance(node, ast.ImportFrom):
            module = node.module or ""
            for alias in node.names:
                imports.append(f"Import: from {module} import {alias.name}")
    
    if imports:
        results.append("=== IMPORTS ===")
        results.extend(imports[:10])  # limit to 10 imports
    
    # 2. Extract classes and their methods
    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef):
            class_name = node.name
            doc = ast.get_docstring(node)
            
            class_info = f"Class: {class_name}"
            if doc:
                class_info += f"\nDescription: {doc}"
            
            # Find class methods
            methods = []
            for child in ast.walk(node):
                if isinstance(child, ast.FunctionDef):
                    methods.append(f"  Method: {child.name}")
            
            if methods:
                class_info += "\nMethods:\n" + "\n".join(methods[:5])
            
            results.append(class_info)
    
    # 3. Extract functions with full signatures
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            name = node.name
            
            # Skip dunder methods (__init__, __str__, etc.) - optional
            if name.startswith('__') and name.endswith('__'):
                continue
            
            # Extract parameters
            args = []
            for arg in node.args.args:
                arg_name = arg.arg
                # Type annotation if present
                if arg.annotation:
                    arg_type = ast.unparse(arg.annotation)
                    args.append(f"{arg_name}: {arg_type}")
                else:
                    args.append(arg_name)
            
            # Return type annotation
            return_type = ""
            if node.returns:
                return_type = f" -> {ast.unparse(node.returns)}"
            
            signature = f"{name}({', '.join(args)}){return_type}"
            
            # Docstring
            doc = ast.get_docstring(node)
            
            # Decorators
            decorators = []
            for decorator in node.decorator_list:
                dec_name = ast.unparse(decorator)
                decorators.append(dec_name)
            
            # Build result
            summary = f"Function: {signature}"
            
            if decorators:
                summary += f"\nDecorators: {', '.join(decorators)}"
            
            if doc:
                # Take only first line of docstring for brevity
                doc_first_line = doc.split('\n')[0]
                summary += f"\nDescription: {doc_first_line}"
            
            results.append(summary)
    
    # 4. Extract important variables/constants (ALL_CAPS)
    constants = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    var_name = target.id
                    # Constants are typically in UPPER_CASE
                    if var_name.isupper():
                        if isinstance(node.value, ast.Constant):
                            constants.append(f"Constant: {var_name} = {repr(node.value.value)}")
                        else:
                            constants.append(f"Constant: {var_name}")
    
    if constants:
        results.append("=== CONSTANTS ===")
        results.extend(constants[:10])
    
    # 5. Extract main block (if __name__ == "__main__")
    for node in ast.walk(tree):
        if isinstance(node, ast.If):
            if (isinstance(node.test, ast.Compare) and 
                isinstance(node.test.left, ast.Name) and 
                node.test.left.id == '__name__'):
                results.append("=== ENTRY POINT ===")
                results.append("Script has main execution block (if __name__ == '__main__')")
                break
    
    return "\n".join(results)


def extract_comments(code):
    """
    Extracts all comments from Python code
    """
    comments = []
    try:
        tokens = tokenize.generate_tokens(StringIO(code).readline)
        for tok_type, tok_string, start, end, line in tokens:
            if tok_type == tokenize.COMMENT:
                comment = tok_string.strip()
                # Skip shebang and encoding comments
                if not comment.startswith('#!'):
                    comments.append(comment)
    except Exception:
        pass
    
    return comments

def extract_python_semantics_full(code):
    """
    Combines advanced code semantics + comments for maximum information
    """
    # Main code information
    main_info = extract_python_semantics_advanced(code)
    
    # Extract comments
    comments = extract_comments(code)
    
    results = []
    
    if main_info:
        results.append(main_info)
    
    if comments:
        results.append("\n=== COMMENTS ===")
        # Only take useful comments (not empty, not too short)
        useful_comments = [c for c in comments if len(c) > 10 and not c.startswith('# TODO')]
        results.extend(useful_comments[:15])
    
    return "\n".join(results)

### Process notebook cells

In [10]:
def process_ipynb(content):
    """
    Converts notebook cells into semantic text
    
    Strategy:
    - keep markdown
    - summarize code
    - merge everything into one text block
    """
    text_blocks = []

    for cell in content:
        if cell["type"] == "markdown":
            text_blocks.append(cell["text"])

        elif cell["type"] == "code":
            summary = extract_python_semantics_full(cell["text"])
            if summary:
                text_blocks.append(summary)

    return "\n".join(text_blocks)

### Process Python files

In [12]:
def process_py(content):
    """
    Extract semantic info from Python file
    """
    return extract_python_semantics_full(content)

### Process plain text files

In [14]:
def process_text(content):
    """
    Clean markdown / pdf text
    """
    return clean_text(content)

### Chunking

In [16]:
import re

def chunk_text_fallback(text, chunk_size=500, overlap=100):
    """
    Fallback: splits text into overlapping chunks by characters
    Used when semantic splitting is not possible
    
    Parameters:
    -----------
    text : str
        Input text to chunk
    chunk_size : int
        Maximum size of each chunk in characters
    overlap : int
        Number of overlapping characters between chunks
    
    Returns:
    --------
    list of str
        List of text chunks
    """
    chunks = []
    
    if not text:
        return chunks
    
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)
    
    return chunks


def smart_chunk_text(text, chunk_size=500, overlap=100):
    """
    Primary: splits text by sentence boundaries (. ! ?)
    Preserves sentence and paragraph integrity
    
    Parameters:
    -----------
    text : str
        Input text to chunk
    chunk_size : int
        Maximum size of each chunk in characters
    overlap : int
        Number of overlapping characters between chunks
    
    Returns:
    --------
    list of str
        List of text chunks with preserved sentence boundaries
    """
    if not text:
        return []
    
    # If text is short, return as single chunk
    if len(text) <= chunk_size:
        return [text]
    
    # Step 1: Split by sentence boundaries
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    # Step 2: Check if semantic splitting is possible
    # If very few sentences or sentences are too short -> use fallback
    if len(sentences) < 2 or max(len(s) for s in sentences) < 30:
        return chunk_text_fallback(text, chunk_size, overlap)
    
    chunks = []
    current_chunk = []
    current_length = 0
    
    for sentence in sentences:
        sentence_length = len(sentence) + 1  # +1 for trailing space
        
        # Handle single sentence that's longer than chunk_size
        if sentence_length > chunk_size:
            # Save current accumulated chunk if exists
            if current_chunk:
                chunks.append(' '.join(current_chunk))
                current_chunk = []
                current_length = 0
            
            # Split the long sentence using fallback method
            sub_chunks = chunk_text_fallback(sentence, chunk_size, overlap)
            chunks.extend(sub_chunks)
            continue
        
        # Check if adding this sentence would exceed chunk_size
        if current_length + sentence_length > chunk_size and current_chunk:
            # Save current chunk
            chunks.append(' '.join(current_chunk))
            
            # Add overlap (last 1-2 sentences)
            overlap_count = 2 if len(current_chunk) > 2 else len(current_chunk)
            overlap_sentences = current_chunk[-overlap_count:] if overlap_count > 0 else []
            
            current_chunk = overlap_sentences.copy()
            current_length = sum(len(s) + 1 for s in current_chunk)
        
        # Add sentence to current chunk
        current_chunk.append(sentence)
        current_length += sentence_length
    
    # Add the last chunk
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    return chunks


def chunk_text(text, chunk_size=500, overlap=100):
    """
    Unified chunking function - automatically chooses best method
    
    This is the MAIN function you should call from your pipeline.
    It intelligently selects between semantic and fallback chunking.
    
    Parameters:
    -----------
    text : str
        Input text to chunk
    chunk_size : int
        Maximum size of each chunk in characters (default: 500)
    overlap : int
        Number of overlapping characters between chunks (default: 100)
    
    Returns:
    --------
    list of str
        List of text chunks
    """
    # Clean text first (remove extra whitespace)
    text = clean_text(text) if 'clean_text' in dir() else text.strip()
    
    if not text:
        return []
    
    # Try smart chunking first
    chunks = smart_chunk_text(text, chunk_size, overlap)
    
    # If result is empty or only 1 chunk but text is long -> use fallback
    if len(chunks) <= 1 and len(text) > chunk_size:
        chunks = chunk_text_fallback(text, chunk_size, overlap)
    
    return chunks

### Simple classification

In [18]:
def classify_text(text):
    """
    Assigns semantic type to chunk
    """
    text = text.lower()

    if "model" in text:
        return "model"
    elif "pipeline" in text:
        return "pipeline"
    elif "feature" in text:
        return "feature_engineering"
    else:
        return "general"

### Main pipeline

In [20]:
chunks = []

for item in tqdm(parsed_data):
    
    file_type = item["type"]
    project = item["project"]
    content = item["content"]
    
    # Process based on type
    if file_type == "ipynb":
        text = process_ipynb(content)
    
    elif file_type == "py":
        text = process_py(content)
    
    elif file_type in ["md", "pdf"]:
        text = process_text(content)
    
    else:
        continue

    text = clean_text(text)

    if not text:
        continue

    # Chunking
    text_chunks = text_chunks = chunk_text(text, chunk_size=500, overlap=50)

    for chunk in text_chunks:
        chunks.append({
            "text": chunk,
            "project": project,
            "type": classify_text(chunk)
        })

  0%|                                                                                          | 0/291 [00:00<?, ?it/s]<unknown>:7: SyntaxWarning: invalid escape sequence '\w'
<unknown>:7: SyntaxWarning: invalid escape sequence '\w'
<unknown>:8: SyntaxWarning: invalid escape sequence '\d'
<unknown>:12: SyntaxWarning: invalid escape sequence '\d'
<unknown>:13: SyntaxWarning: invalid escape sequence '\-'
<unknown>:14: SyntaxWarning: invalid escape sequence '\['
<unknown>:7: SyntaxWarning: invalid escape sequence '\w'
<unknown>:7: SyntaxWarning: invalid escape sequence '\w'
<unknown>:8: SyntaxWarning: invalid escape sequence '\d'
<unknown>:12: SyntaxWarning: invalid escape sequence '\d'
<unknown>:13: SyntaxWarning: invalid escape sequence '\-'
<unknown>:14: SyntaxWarning: invalid escape sequence '\['
  5%|████▍                                                                           | 16/291 [00:00<00:01, 151.18it/s]<unknown>:3: SyntaxWarning: invalid escape sequence '\d'
<unknown>:10: S

### Save result

In [22]:
OUTPUT_PATH = "../data/chunks/chunks.json"

import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"Saved {len(chunks)} chunks")

Saved 8361 chunks


### Summary

- Loaded parsed data from the previous step (`parsed_data.json`)  
- Converted raw content into **semantic text representations**  
  - extracted meaning from Python code (functions, docstrings)  
  - combined notebook markdown + code into unified text  
- Cleaned and normalized text  
- Split text into **overlapping chunks** (chunking)  
- Added metadata (project, type) to each chunk  

👉 Result: structured **semantic chunks** ready for embeddings and retrieval (RAG pipeline)